In [1]:
from dask.distributed import Client
from dask.dataframe import from_pandas, read_csv, merge
import pandas as pd
import requests
from datetime import date, timedelta

In [ ]:
client = Client(n_workers=8, threads_per_worker=1, memory_limit='1GB')
print(client.dashboard_link)

http://127.0.0.1:8787/status


2026-03-06 14:20:59,467 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:40361' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {('frompandas-3b9dc864d3d4f8b3b7e808ca06356621', 31), ('frompandas-3b9dc864d3d4f8b3b7e808ca06356621', 33)} (stimulus_id='handle-worker-cleanup-1772835659.4647343')
2026-03-06 14:20:59,479 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:42711' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {('frompandas-3b9dc864d3d4f8b3b7e808ca06356621', 49), ('frompandas-3b9dc864d3d4f8b3b7e808ca06356621', 38)} (stimulus_id='handle-worker-cleanup-1772835659.4789612')
2026-03-06 14:20:59,481 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:36649' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {('frompandas-3b9dc864d3d4f8b3b7e808ca06356621', 58), ('frompandas-3b9dc864d3d4f8b3b7e808ca06356621', 47)

In [3]:
calendar_df = []

for day_of_year in range(1, 366):
    for year in range(2020, 2027):
        calendar_df.append({'day_of_year': day_of_year, 'year': year})

calendar_df = pd.DataFrame(calendar_df)
calendar_df.head()

,day_of_year,year
0,1,2020
1,1,2021
2,1,2022
3,1,2023
4,1,2024


In [4]:
ski_resort_df = pd.read_csv('/media/kwierman/Data/powderpipeline/ski_resorts.csv')

ski_resort_df.head()

,Resort_Name,Pass_Affiliation,Latitude,Longitude,Base_Elevation_ft,Summit_Elevation_ft
0,Vail,Epic,39.6403,-106.3742,8120,11570
1,Breckenridge,Epic,39.4808,-106.0667,9600,12998
2,Keystone,Epic,39.6053,-105.9515,9280,12408
3,Crested Butte,Epic,38.8997,-106.9658,9375,12162
4,Beaver Creek,Epic,39.6042,-106.5165,8100,11440


In [5]:
df = pd.merge(ski_resort_df, calendar_df, how='cross')
df.head()


,Resort_Name,Pass_Affiliation,Latitude,Longitude,Base_Elevation_ft,Summit_Elevation_ft,day_of_year,year
0,Vail,Epic,39.6403,-106.3742,8120,11570,1,2020
1,Vail,Epic,39.6403,-106.3742,8120,11570,1,2021
2,Vail,Epic,39.6403,-106.3742,8120,11570,1,2022
3,Vail,Epic,39.6403,-106.3742,8120,11570,1,2023
4,Vail,Epic,39.6403,-106.3742,8120,11570,1,2024


In [6]:
ddf = from_pandas(df, npartitions=100)

ddf

,Resort_Name,Pass_Affiliation,Latitude,Longitude,Base_Elevation_ft,Summit_Elevation_ft,day_of_year,year
npartitions=100,,,,,,,,
0,string,string,float64,float64,int64,int64,int64,int64
1942,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...
192239,...,...,...,...,...,...,...,...
194179,...,...,...,...,...,...,...,...


In [7]:



def get_snow_conditions(lat, lon, elevation_ft, day_of_year, year=2024):
    """
    Fetches the total snowfall and average snow depth for a specific location, 
    elevation, and day of the year using the Open-Meteo Archive API.
    
    Parameters:
    - lat (float): Latitude of the ski resort
    - lon (float): Longitude of the ski resort
    - elevation_ft (float): Elevation in feet (tool converts it to meters for the API)
    - day_of_year (int): Day of the year (1 to 365/366)
    - year (int): The historical year to query. Defaults to 2024.
    
    Returns:
    - dict: A dictionary containing snowfall and snow depth in inches.
    """
    
    # 1. Convert elevation from feet to meters (required by the API)
    elevation_m = int(elevation_ft * 0.3048)
    
    # 2. Convert "Day of the Year" into a specific Date string (YYYY-MM-DD)
    # We use Jan 1st of the specified year and add the day_of_year minus 1
    target_date = date(year, 1, 1) + timedelta(days=day_of_year - 1)
    date_str = target_date.strftime("%Y-%m-%d")
    
    # 3. Setup the Open-Meteo API request
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "elevation": elevation_m,
        "start_date": date_str,
        "end_date": date_str,
        "hourly": "snowfall,snow_depth", # Fetching hourly to aggregate accurately
        "timezone": "auto"
    }
    
    try:
        # 4. Make the API Call
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()
        
        # 5. Extract the hourly data
        hourly_snowfall_cm = data.get("hourly", {}).get("snowfall",[])
        hourly_snow_depth_m = data.get("hourly", {}).get("snow_depth",[])
        
        # Filter out any missing (None) values
        hourly_snowfall_cm =[s for s in hourly_snowfall_cm if s is not None]
        hourly_snow_depth_m =[d for d in hourly_snow_depth_m if d is not None]
        
        # 6. Aggregate the data
        # Snowfall is cumulative, so we sum the hourly amounts
        total_snowfall_cm = sum(hourly_snowfall_cm) if hourly_snowfall_cm else 0.0
        # Snow depth is a standing measurement, so we take the daily maximum
        max_snow_depth_m = max(hourly_snow_depth_m) if hourly_snow_depth_m else 0.0
        
        # 7. Convert from Metric back to Imperial (Inches) for U.S. standards
        total_snowfall_in = total_snowfall_cm * 0.393701
        max_snow_depth_in = max_snow_depth_m * 39.3701
        
        return {
            "query_date": date_str,
            "day_of_year": day_of_year,
            "total_snowfall_inches": round(total_snowfall_in, 2),
            "snow_depth_inches": round(max_snow_depth_in, 2)
        }
        
    except requests.exceptions.RequestException as e:
        print(f"Failed to fetch data from API. Error: {e}")
        return None


In [8]:
def get_snow_conditions_for_resort(row):
    base_conditions = get_snow_conditions(
        lat=row['Latitude'],
        lon=row['Longitude'],
        elevation_ft=row['Base_Elevation_ft'],
        day_of_year=row['day_of_year'],
        year=row['year']
    )
    summit_conditions = get_snow_conditions(
        lat=row['Latitude'],
        lon=row['Longitude'],
        elevation_ft=row['Summit_Elevation_ft'],
        day_of_year=row['day_of_year'],
        year=row['year']
    )

    row['base_total_snowfall_in'] = base_conditions['total_snowfall_inches'] if base_conditions else None
    row['base_snow_depth_in'] = base_conditions['snow_depth_inches'] if base_conditions else None
    row['summit_total_snowfall_in'] = summit_conditions['total_snowfall_inches'] if summit_conditions else None
    row['summit_snow_depth_in'] = summit_conditions['snow_depth_inches'] if summit_conditions else None
    return row


In [9]:
meta = pd.DataFrame({'Resort_Name': pd.Series(dtype='string'), 
                     'Pass_Affiliation': pd.Series(dtype='string'),
                     'Latitude': pd.Series(dtype='float64'),
                     'Longitude': pd.Series(dtype='float64'),
                     'Base_Elevation_ft': pd.Series(dtype='int64'),
                     'Summit_Elevation_ft': pd.Series(dtype='int64'),
                     'day_of_year': pd.Series(dtype='int64'),
                     'year': pd.Series(dtype='int64'),
                     'base_total_snowfall_in': pd.Series(dtype='float64'),
                     'base_snow_depth_in': pd.Series(dtype='float64'),
                     'summit_total_snowfall_in': pd.Series(dtype='float64'),
                     'summit_snow_depth_in': pd.Series(dtype='float64')}) # adjust dtypes as needed
ddf = ddf.apply(get_snow_conditions_for_resort, axis=1, meta=meta)

In [10]:
ddf

,Resort_Name,Pass_Affiliation,Latitude,Longitude,Base_Elevation_ft,Summit_Elevation_ft,day_of_year,year,base_total_snowfall_in,base_snow_depth_in,summit_total_snowfall_in,summit_snow_depth_in
npartitions=100,,,,,,,,,,,,
0,string,string,float64,float64,int64,int64,int64,int64,float64,float64,float64,float64
1942,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...
192239,...,...,...,...,...,...,...,...,...,...,...,...
194179,...,...,...,...,...,...,...,...,...,...,...,...


In [ ]:
ddf.to_parquet('/media/kwierman/Data/powderpipeline/ski_resort_snow_conditions.parquet', overwrite=True)

/home/kwierman/Desktop/Projects/powderpipeline/.venv/lib/python3.12/site-packages/distributed/client.py:3387: UserWarning: Sending large graph of size 14.52 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


Failed to fetch data from API. Error: 429 Client Error: Too Many Requests for url: https://archive-api.open-meteo.com/v1/archive?latitude=43.5875&longitude=-110.8279&elevation=1923&start_date=2021-05-28&end_date=2021-05-28&hourly=snowfall%2Csnow_depth&timezone=auto
Failed to fetch data from API. Error: 429 Client Error: Too Many Requests for url: https://archive-api.open-meteo.com/v1/archive?latitude=44.3658&longitude=-73.8665&elevation=371&start_date=2024-03-14&end_date=2024-03-14&hourly=snowfall%2Csnow_depth&timezone=auto
Failed to fetch data from API. Error: 400 Client Error: Bad Request for url: https://archive-api.open-meteo.com/v1/archive?latitude=39.5017&longitude=-106.1564&elevation=2960&start_date=2026-03-30&end_date=2026-03-30&hourly=snowfall%2Csnow_depth&timezone=auto
Failed to fetch data from API. Error: HTTPSConnectionPool(host='archive-api.open-meteo.com', port=443): Read timed out. (read timeout=None)
Failed to fetch data from API. Error: HTTPSConnectionPool(host='archiv

2026-03-06 14:20:59,417 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute(('apply-2dcdb45ff8c527569db3def61540dad2', 49))" coro=<Worker.execute() done, defined at /home/kwierman/Desktop/Projects/powderpipeline/.venv/lib/python3.12/site-packages/distributed/worker_state_machine.py:3608>> ended with CancelledError
2026-03-06 14:20:59,417 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute(('apply-2dcdb45ff8c527569db3def61540dad2', 58))" coro=<Worker.execute() done, defined at /home/kwierman/Desktop/Projects/powderpipeline/.venv/lib/python3.12/site-packages/distributed/worker_state_machine.py:3608>> ended with CancelledError
2026-03-06 14:20:59,419 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute(('apply-2dcdb45ff8c527569db3def61540dad2', 88))" coro=<Worker.execute() done, defined at /home/kwierman/Desktop/Projects/powderpipeline/.venv/lib/pytho

Failed to fetch data from API. Error: 429 Client Error: Too Many Requests for url: https://archive-api.open-meteo.com/v1/archive?latitude=41.216&longitude=-111.8569&elevation=1950&start_date=2023-04-22&end_date=2023-04-22&hourly=snowfall%2Csnow_depth&timezone=auto
Failed to fetch data from API. Error: 429 Client Error: Too Many Requests for url: https://archive-api.open-meteo.com/v1/archive?latitude=43.5875&longitude=-110.8279&elevation=1923&start_date=2025-08-21&end_date=2025-08-21&hourly=snowfall%2Csnow_depth&timezone=auto
Failed to fetch data from API. Error: 429 Client Error: Too Many Requests for url: https://archive-api.open-meteo.com/v1/archive?latitude=44.9367&longitude=-72.4545&elevation=1209&start_date=2022-04-22&end_date=2022-04-22&hourly=snowfall%2Csnow_depth&timezone=autoFailed to fetch data from API. Error: 429 Client Error: Too Many Requests for url: https://archive-api.open-meteo.com/v1/archive?latitude=48.4795&longitude=-114.3503&elevation=2077&start_date=2025-04-26&en

KeyboardInterrupt: 

Failed to fetch data from API. Error: 429 Client Error: Too Many Requests for url: https://archive-api.open-meteo.com/v1/archive?latitude=36.596&longitude=-105.4545&elevation=2806&start_date=2023-06-22&end_date=2023-06-22&hourly=snowfall%2Csnow_depth&timezone=auto
Failed to fetch data from API. Error: 429 Client Error: Too Many Requests for url: https://archive-api.open-meteo.com/v1/archive?latitude=43.5875&longitude=-110.8279&elevation=1923&start_date=2020-08-21&end_date=2020-08-21&hourly=snowfall%2Csnow_depth&timezone=auto
Failed to fetch data from API. Error: 429 Client Error: Too Many Requests for url: https://archive-api.open-meteo.com/v1/archive?latitude=41.216&longitude=-111.8569&elevation=1950&start_date=2025-04-22&end_date=2025-04-22&hourly=snowfall%2Csnow_depth&timezone=auto
Failed to fetch data from API. Error: 429 Client Error: Too Many Requests for url: https://archive-api.open-meteo.com/v1/archive?latitude=48.4795&longitude=-114.3503&elevation=2077&start_date=2020-04-26&e

In [ ]:
client.close()

2026-03-06 10:53:21,517 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute(('apply-f86fa986b8405945ead22017f806bab8', 645))" coro=<Worker.execute() done, defined at /home/kwierman/Desktop/Projects/powderpipeline/.venv/lib/python3.12/site-packages/distributed/worker_state_machine.py:3608>> ended with CancelledError
2026-03-06 10:53:21,517 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute(('apply-f86fa986b8405945ead22017f806bab8', 910))" coro=<Worker.execute() done, defined at /home/kwierman/Desktop/Projects/powderpipeline/.venv/lib/python3.12/site-packages/distributed/worker_state_machine.py:3608>> ended with CancelledError
2026-03-06 10:53:21,517 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute(('apply-f86fa986b8405945ead22017f806bab8', 474))" coro=<Worker.execute() done, defined at /home/kwierman/Desktop/Projects/powderpipeline/.venv/lib/py

Resort_Name                 Mt. Bachelor
Pass_Affiliation                    Ikon
Latitude                         43.9889
Longitude                      -121.6818
Base_Elevation_ft                   6300
Summit_Elevation_ft                 9065
day_of_year                           26
year                                2023
base_total_snowfall_in               0.0
base_snow_depth_in                 34.65
summit_total_snowfall_in             0.0
summit_snow_depth_in               34.65
Name: 92158, dtype: object
Resort_Name                 Smugglers Notch
Pass_Affiliation                       <NA>
Latitude                            44.5885
Longitude                            -72.79
Base_Elevation_ft                      1030
Summit_Elevation_ft                    3640
day_of_year                              64
year                                   2026
base_total_snowfall_in                  0.0
base_snow_depth_in                    23.62
summit_total_snowfall_in                0

2026-03-06 10:53:25,517 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2026-03-06 10:53:25,518 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2026-03-06 10:53:25,518 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
